This program computes the spherical Bessel Functions $j_l(x)$ for $l = 0, \dots, 25$ and $x = 0.1, 1, 10$ using upward and downward recursion.
The goal is to demonstrate that the stability of recurrence relations can depend on the direction you apply them.

In [6]:
import numpy as np
from scipy.special import spherical_jn # reference values

In [7]:
def j_upward(x, lmax):
    """Compute j_l(x) for l=0 to lmax using upward recursion."""
    j = np.zeros(lmax + 1)

    # Exact starting values
    if x == 0:
        j[0] = 1.0
        j[1] = 0.0
    else:
        j[0] = np.sin(x)/x
        j[1] = np.sin(x) / (x * x) - np.cos(x)/x
    for l in range(1, lmax):
        j[l+1] = (2*l + 1) / x * j[l] - j[l-1]
    return j

In [8]:
def j_downward(x, lmax, L_start=None):
    """
    Compute j_l(x) for l=0...lmax using downward recursion (Miller's algorithm).
    L_start is the stearting index; if None it is chosen heuristically.
    """
    if L_start is None:
        # Choose L_start large enough: at least lmax+20 and >2*x
        L_start = max(lmax + 20, int(2*x) + 20, 50)
    j = np.zeros(L_start + 1)
    # initial guess for high orders
    j[L_start] = 0.0
    j[L_start - 1] = 1.0
    # recurse downward
    for l in range(L_start - 1, 0, -1):
        j[l-1] =(2*l + 1)/x * j[l] - j[l+1]
    # normalize using exact j0
    j0_exact = np.sin(x) / x if x != 0 else 1.0
    norm = j0_exact / j[0]
    return j[:lmax+1] * norm

In [9]:
#parameters 
l_max = 25
x_vals = [0.1, 1.0, 10.0]

for x in x_vals:
    print(f"\nResults for x = {x}")

    # Compute using both methods
    j_up = j_upward(x, l_max)
    j_down = j_downward(x, l_max)
    j_ref = np.array([spherical_jn(l, x) for l in range(l_max+1)])

    print(f"{'l':>3} {'j_up':>20} {'j_down':>20} {'rel_diff':>15} {'rel_err_up':>15} {'rel_err_down':>15}")

    for l in range(l_max+1):
        # relative differences up and down
        denom = abs(j_up[l]) + abs(j_down[l])
        rel_diff = abs(j_up[l] - j_down[l]) / denom if denom != 0 else 0.0

        # relative error w.r.t. reference
        ref = j_ref[l]
        if abs(ref) > 1e-15:
            rel_err_up = abs(j_up[l] - ref) / abs(ref)
            rel_err_down = abs(j_down[l] - ref) / abs(ref)
        else:
            rel_err_up = abs(j_up[l] - ref)
            rel_err_down = abs(j_down[l] - ref)

        print(f"{l:3d} {j_up[l]:20.12e} {j_down[l]:20.12e}"
              f"{rel_diff:15.5e} {rel_err_up:15.5e} {rel_err_down:15.5e}")


    # Convergence study for downward recursion:
    print("\nConvergence of downward recursion (max relative change for l<=25):")
    L_tests = [30, 40, 50, 60, 80]
    prev = None
    for L in L_tests:
        j_down_test = j_downward(x, l_max, L_start=L)
        if prev is not None:
            # maximum relative change between successive L
            diff = np.abs(j_down_test - prev)
            denom = np.abs(j_down_test) + np.abs(prev)
            max_change = np.max(diff / denom) if  np.any(denom > 0) else 0.0
            print(f" L_start={L:3d} : max relative change = {max_change:.2e}")
        prev = j_down_test

    # Stability of upward recursion: show error growth
    print("\n Stability of upward recursion (relative error vs l):")
    for l in range(0, l_max+1, 5):
        if abs(j_ref[l]) > 1e-15:
            err = abs(j_up[l] - j_ref[l]) / abs(j_ref[l])
        else:
            err = abs(j_up[l] - j_ref[l])
        print(f" l={l:2d} : rel.error = {err:.2e}")


Results for x = 0.1
  l                 j_up               j_down        rel_diff      rel_err_up    rel_err_down
  0   9.983341664683e-01   9.983341664683e-01    0.00000e+00     0.00000e+00     0.00000e+00
  1   3.330001190256e-02   3.330001190256e-02    2.45883e-14     5.04268e-14     1.25025e-15
  2   6.661906083966e-04   6.661906084456e-04    3.67781e-11     7.35572e-11     9.76479e-16
  3   9.518517272378e-06   9.518519720866e-06    1.28617e-07     2.57234e-07     1.42381e-15
  4   1.056006698752e-07   1.057720150210e-07    8.10631e-04     1.61995e-03     2.87791e-15
  5  -1.445698361024e-08   9.616310232916e-10    1.00000e+00     1.60338e+01     4.30093e-16
  6  -1.695868867002e-06   7.397541093588e-12    1.00000e+00     2.29249e+05     6.55185e-16
  7  -2.204484957267e-04   4.931887475732e-14    1.00000e+00     4.46986e+09     1.79145e-15
  8  -3.306557849013e-02   2.901200102530e-16    1.00000e+00     3.30656e-02     3.94430e-31
  9  -5.620927894827e+00   1.526985693495e-18   

In [10]:
print("\nExplanation:")
print("When x is large (e.g., x=10), upward recursion is stable for l < x (here l < 10),")
print("while downward recursion is stable for all l. For l < x, both methods produce")
print("accurate values, leading to very small relative differences. For x=1, only l=0")
print("lies below x, so only that index shows agreement. For x=0.1, no l > 0 is below x,")
print("so the two methods disagree for l≥2. The region of agreement is precisely where")
print("the upward recurrence remains stable, i.e., l ≲ x.")


Explanation:
When x is large (e.g., x=10), upward recursion is stable for l < x (here l < 10),
while downward recursion is stable for all l. For l < x, both methods produce
accurate values, leading to very small relative differences. For x=1, only l=0
lies below x, so only that index shows agreement. For x=0.1, no l > 0 is below x,
so the two methods disagree for l≥2. The region of agreement is precisely where
the upward recurrence remains stable, i.e., l ≲ x.
